# Topic Modeling pe Wikipedia: Persoana B (NMF + LDA)

Acest notebook acoperă **doar partea persoanei B**, pornind de la fișierele exportate de persoana A în `shared/`:

1. Încărcarea datelor, matricilor BoW / TF-IDF și a corpusului de referință Gensim
2. **NMF** și **LDA** (scikit-learn) pe BoW și pe TF-IDF, pentru **k = 3 și k = 5**:
   * top 10 cuvinte per topic
   * atribuirea documentelor la topicul dominant
   * verificarea dacă documentele din același domeniu sunt grupate
   * scatter plot-uri (cuvinte + documente)
   * scoruri de coerență Gensim
3. Răspunsurile la cele 5 întrebări din Partea 4, **doar pentru NMF și LDA**

In [1]:
%pip install -q scikit-learn gensim numpy pandas scipy plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score

IN_DIR = Path("shared")                  # primit de la persoana A
OUT_DIR = Path("shared")                 # rezultatele lui B se salvează lângă cele ale lui A
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
SHOW_PLOTS = True                        # False = doar salvează graficele ca HTML

# ---- parametri comuni cu persoana A (citiți din config.json, nu se schimbă) ----
cfg = json.load(open(IN_DIR / "config.json"))
RANDOM_STATE = cfg["random_state"]
K_VALUES = tuple(cfg["k_values"])
N_TOP_WORDS = cfg["n_top_words"]

## 0. Încărcarea datelor partajate

In [3]:
df = pd.read_csv(IN_DIR / "corpus.csv")
X_bow = sparse.load_npz(IN_DIR / "bow.npz")
X_tfidf = sparse.load_npz(IN_DIR / "tfidf.npz")
feats = json.load(open(IN_DIR / "vocab.json"))
chunk_texts = json.load(open(IN_DIR / "chunks.json"))
dictionary = Dictionary.load(str(IN_DIR / "gensim_dictionary.dict"))

titles = df["title"].tolist()
subjects = df["subject"].tolist()
classes = sorted(set(subjects))
true_idx = np.array([classes.index(s) for s in subjects])
MATRICES = {"BoW": X_bow, "TF-IDF": X_tfidf}

assert X_bow.shape == X_tfidf.shape == (len(df), len(feats))
print(f"{len(df)} documente | domenii: {classes}")
print(f"BoW {X_bow.shape} | TF-IDF {X_tfidf.shape} | vocabular {len(feats)} cuvinte")
print(f"Corpus de referință Gensim: {len(chunk_texts)} pseudo-paragrafe, {len(dictionary)} cuvinte")
print(f"random_state={RANDOM_STATE}, k={K_VALUES}, top {N_TOP_WORDS} cuvinte")

30 documente | domenii: ['animals', 'green_energy', 'roman_empire']
BoW (30, 5854) | TF-IDF (30, 5854) | vocabular 5854 cuvinte
Corpus de referință Gensim: 1119 pseudo-paragrafe, 13060 cuvinte
random_state=42, k=(3, 5), top 10 cuvinte


## 1. NMF și LDA pe BoW și TF-IDF, k = 3 și k = 5

* **NMF** factorizează `X ≈ W·H` cu toate valorile ≥ 0. `W` (documente × topicuri) spune cât din fiecare topic conține un document, `H` (topicuri × cuvinte) spune ce cuvinte definesc topicul. Fiind non-negative, topicurile sunt aditive și, de obicei, ușor de citit (spre deosebire de LSA, fără semne mixte). Inițializare `nndsvd` (deterministă) și `random_state` comun.
* **LDA** este model probabilistic generativ: fiecare document este un amestec de topicuri, fiecare topic o distribuție peste cuvinte. `transform` dă distribuția topicurilor per document (`θ`), `components_` ponderile cuvintelor per topic (le normalizăm la probabilități). LDA lucrează natural cu **numărări** (BoW); pe TF-IDF valorile sunt fracționare, dar scikit-learn le acceptă, așa că rulăm și această combinație cum cere tema.

Pentru fiecare combinație se păstrează `doc_topics` (documente × k) și `components` (k × cuvinte), ca să se poată folosi aceleași funcții pentru atribuire, grafice și coerență.

In [4]:
def top_idx(components, n=N_TOP_WORDS):
    return [np.argsort(row)[::-1][:n] for row in components]


def run_nmf(X, name, k):
    m = NMF(n_components=k, init="nndsvd", max_iter=1000, random_state=RANDOM_STATE)
    W = m.fit_transform(X)
    return dict(name=name, k=k, doc_topics=W, components=m.components_, top_idx=top_idx(m.components_),
                extra=f"eroare reconstrucție = {m.reconstruction_err_:.3f}")


def run_lda(X, name, k):
    m = LatentDirichletAllocation(n_components=k, learning_method="batch", max_iter=100,
                                  random_state=RANDOM_STATE)
    theta = m.fit_transform(X)                                      # distribuția topicurilor / document
    comp = m.components_ / m.components_.sum(axis=1, keepdims=True)  # P(cuvânt | topic)
    return dict(name=name, k=k, doc_topics=theta, components=comp, top_idx=top_idx(comp),
                extra=f"perplexitate = {m.perplexity(X):.1f}")


def topic_words(res):
    return [[feats[i] for i in idx] for idx in res["top_idx"]]


results = {}
for method, runner in (("NMF", run_nmf), ("LDA", run_lda)):
    for k in K_VALUES:
        for mname, X in MATRICES.items():
            name = f"{method} ({mname}, k={k})"
            results[name] = runner(X, name, k)

for name, res in results.items():
    print(f"\n=== {name} | {res['extra']} ===")
    for j, words in enumerate(topic_words(res)):
        print(f"  Topic {j}: {', '.join(words)}")


=== NMF (BoW, k=3) | eroare reconstrucție = 1201.911 ===
  Topic 0: roman, augustus, empire, rome, emperor, hadrian, diocletian, octavian, imperial, army
  Topic 1: wolf, bear, lion, polar, prey, specie, eagle, male, tiger, female
  Topic 2: energy, solar, wind, renewable, electricity, hydrogen, cost, fuel, capacity, country

=== NMF (TF-IDF, k=3) | eroare reconstrucție = 4.725 ===
  Topic 0: roman, empire, emperor, army, hadrian, rome, diocletian, military, augustus, constantine
  Topic 1: energy, solar, wind, electricity, renewable, geothermal, cost, tidal, hydrogen, fuel
  Topic 2: tiger, lion, elephant, dolphin, bear, panda, eagle, polar, gorilla, wolf

=== NMF (BoW, k=5) | eroare reconstrucție = 1066.612 ===
  Topic 0: roman, augustus, empire, rome, emperor, hadrian, diocletian, octavian, imperial, army
  Topic 1: wolf, dog, pack, prey, kill, hunt, human, attack, specie, north
  Topic 2: energy, solar, wind, renewable, electricity, hydrogen, cost, fuel, capacity, country
  Topic 

## 2. Atribuirea documentelor la topicul dominant

Fiecare document merge la topicul cu **cel mai mare scor** (`argmax` pe `W` la NMF, pe `θ` la LDA; scorurile sunt non-negative, deci nu mai e nevoie de valoare absolută ca la LSA).

Pentru a vedea dacă documentele din același domeniu sunt grupate:
* **puritatea**: fracția documentelor care aparțin domeniului majoritar al topicului în care au fost puse (1.0 = fiecare topic conține un singur domeniu);
* **ARI** (Adjusted Rand Index) față de cele 3 domenii reale (1.0 = partiție identică; rămâne comparabil și la k=5);
* **cel mai mare grup**: mărimea celui mai mare topic (detectează colapsul tuturor documentelor într-un singur topic);
* ca variantă de comparație, KMeans pe vectorii documentelor normalizați (aceeași metodă ca la A).

In [5]:
def dominant(res):
    return res["doc_topics"].argmax(axis=1)


def dominant_kmeans(res):
    D = res["doc_topics"]
    Dn = D / np.clip(np.linalg.norm(D, axis=1, keepdims=True), 1e-12, None)
    return KMeans(n_clusters=res["k"], n_init=10, random_state=RANDOM_STATE).fit_predict(Dn)


def purity(pred):
    return sum(np.bincount(true_idx[pred == c], minlength=len(classes)).max()
               for c in np.unique(pred)) / len(pred)


assignments, rows = {}, []
for name, res in results.items():
    p, p_km = dominant(res), dominant_kmeans(res)
    assignments[name] = p
    rows.append({"model": name, "purity": purity(p), "ARI": adjusted_rand_score(true_idx, p),
                 "cel mai mare grup": int(np.bincount(p).max()),
                 "purity (kmeans)": purity(p_km), "ARI (kmeans)": adjusted_rand_score(true_idx, p_km)})
assign_summary = pd.DataFrame(rows).set_index("model")
assign_summary.round(3)

,purity,ARI,cel mai mare grup,purity (kmeans),ARI (kmeans)
model,,,,,
"NMF (BoW, k=3)",1.000,1.000,10,1.000,1.000
"NMF (TF-IDF, k=3)",1.000,1.000,10,1.000,1.000
"NMF (BoW, k=5)",1.000,0.870,10,1.000,0.870
"NMF (TF-IDF, k=5)",1.000,0.708,10,1.000,0.708
"LDA (BoW, k=3)",1.000,1.000,10,1.000,1.000
"LDA (TF-IDF, k=3)",0.767,0.498,15,0.767,0.498
"LDA (BoW, k=5)",1.000,0.766,10,1.000,0.766
"LDA (TF-IDF, k=5)",0.333,0.000,30,0.600,0.053


In [6]:
# Documentele din fiecare domeniu, pe topicurile în care au fost puse
for name, pred in assignments.items():
    print(f"\n=== {name} ===")
    print(pd.crosstab(pd.Series(np.array(subjects), name="domeniu"), pd.Series(pred, name="topic")))

assign_df = pd.DataFrame({"title": titles, "subject": subjects, **assignments})
assign_df.to_csv(OUT_DIR / "nmf_lda_assignments_B.csv", index=False)
assign_df


=== NMF (BoW, k=3) ===
topic          0   1   2
domeniu                 
animals        0  10   0
green_energy   0   0  10
roman_empire  10   0   0

=== NMF (TF-IDF, k=3) ===
topic          0   1   2
domeniu                 
animals        0   0  10
green_energy   0  10   0
roman_empire  10   0   0

=== NMF (BoW, k=5) ===
topic          0  1   2  3  4
domeniu                      
animals        0  1   0  2  7
green_energy   0  0  10  0  0
roman_empire  10  0   0  0  0

=== NMF (TF-IDF, k=5) ===
topic         0  1   2  3  4
domeniu                     
animals       0  0  10  0  0
green_energy  0  5   0  0  5
roman_empire  6  0   0  4  0

=== LDA (BoW, k=3) ===
topic          0   1   2
domeniu                 
animals        0  10   0
green_energy   0   0  10
roman_empire  10   0   0

=== LDA (TF-IDF, k=3) ===
topic          0  1   2
domeniu                
animals        2  3   5
green_energy   0  0  10
roman_empire  10  0   0

=== LDA (BoW, k=5) ===
topic          0  1  2  3  4
dome

,title,subject,"NMF (BoW, k=3)","NMF (TF-IDF, k=3)","NMF (BoW, k=5)","NMF (TF-IDF, k=5)","LDA (BoW, k=3)","LDA (TF-IDF, k=3)","LDA (BoW, k=5)","LDA (TF-IDF, k=5)"
0,Solar power,green_energy,2,1,2,1,2,2,4,4
1,Wind power,green_energy,2,1,2,4,2,2,4,4
2,Hydroelectricity,green_energy,2,1,2,4,2,2,4,4
3,Geothermal energy,green_energy,2,1,2,4,2,2,4,4
4,Biofuel,green_energy,2,1,2,1,2,2,3,4
5,Tidal power,green_energy,2,1,2,4,2,2,4,4
6,Renewable energy,green_energy,2,1,2,1,2,2,4,4
7,Wave power,green_energy,2,1,2,4,2,2,4,4
8,Green hydrogen,green_energy,2,1,2,1,2,2,3,4
9,Photovoltaics,green_energy,2,1,2,1,2,2,4,4


## 3. Scatter plot-uri: documente și top cuvinte

* **Documente** (cercuri), colorate după domeniul real, poziționate la `W` (NMF) sau `θ` (LDA).
* **Top 10 cuvinte** ale fiecărui topic (romburi cu etichetă), poziționate la `H` (NMF) sau `P(cuvânt | topic)` (LDA). Culoarea romburilor = topicul din care provin.
* Documentele și cuvintele au scări diferite, deci fiecare set este scalat la maximum 1 pe fiecare axă, ca să încapă pe același grafic (contează pozițiile relative).
* Pentru fiecare model: un grafic 3D pe primele 3 topicuri și grafice 2D pe perechi de topicuri (k=5 are o pereche în plus ca să apară toate topicurile). Se salvează și ca HTML în `shared/plots/`.

In [7]:
SUBJECT_COLORS = ["#1b9e77", "#d95f02", "#7570b3"]
TOPIC_COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]
DIM_SETS = {3: [(0, 1, 2), (1, 2)], 5: [(0, 1, 2), (1, 2), (3, 4)]}


def embed(res):
    idx = np.concatenate(res["top_idx"])
    word_topic = np.repeat(np.arange(res["k"]), N_TOP_WORDS)
    docs = res["doc_topics"] / res["doc_topics"].max(axis=0, keepdims=True).clip(1e-12)
    words = res["components"][:, idx].T
    words = words / words.max(axis=0, keepdims=True).clip(1e-12)
    return docs, words, idx, word_topic


def plot_scatter(res, dims):
    docs, words, idx, word_topic = embed(res)
    tw = topic_words(res)
    three = len(dims) == 3
    Trace = go.Scatter3d if three else go.Scatter
    xyz = lambda M, mask: dict(zip("xyz", (M[mask, d] for d in dims)))
    fig = go.Figure()

    for c, cls in enumerate(classes):
        m = true_idx == c
        fig.add_trace(Trace(**xyz(docs, m), mode="markers", text=[titles[i] for i in np.where(m)[0]],
                            hoverinfo="text", name=f"documente: {cls}",
                            marker=dict(size=6 if three else 10, color=SUBJECT_COLORS[c], opacity=0.85,
                                        line=dict(width=1, color="black"))))
    for j in range(res["k"]):
        m = word_topic == j
        fig.add_trace(Trace(**xyz(words, m), mode="markers+text", text=[feats[i] for i in idx[m]],
                            textposition="top center", hoverinfo="text", name=f"top cuvinte: topic {j}",
                            marker=dict(size=5 if three else 9, symbol="diamond",
                                        color=TOPIC_COLORS[j % len(TOPIC_COLORS)],
                                        line=dict(width=1, color="black"))))

    ax = lambda d: dict(title=f"Topic {d}: {', '.join(tw[d][:3])}")
    layout = dict(title=f"{res['name']}: topicurile {list(dims)}", height=720,
                  legend=dict(itemsizing="constant"), margin=dict(l=0, r=0, t=50, b=0))
    if three:
        layout["scene"] = dict(xaxis=ax(dims[0]), yaxis=ax(dims[1]), zaxis=ax(dims[2]))
    else:
        layout.update(xaxis=ax(dims[0]), yaxis=ax(dims[1]))
    fig.update_layout(**layout)
    return fig


figs = {}
for name, res in results.items():
    for dims in DIM_SETS[res["k"]]:
        fig = plot_scatter(res, dims)
        key = f"{name} {'x'.join(map(str, dims))}"
        figs[key] = fig
        fn = re.sub(r"[^a-z0-9]+", "_", key.lower()).strip("_") + ".html"
        fig.write_html(OUT_DIR / "plots" / fn)
print(f"{len(figs)} grafice salvate în {OUT_DIR / 'plots'}")

20 grafice salvate în shared\plots


In [8]:
if SHOW_PLOTS:
    for key, fig in figs.items():
        fig.show()

## 4. Coerența Gensim

Aceeași procedură ca la persoana A (corpus de referință = pseudo-paragrafe din `shared/`), ca scorurile să fie comparabile:

* `c_v`: coerență pe fereastră glisantă de context + NPMI + similaritate cosinus (mai mare = mai bine, ~0–1)
* `u_mass`: bazată pe co-apariția în documente (valori negative; mai aproape de 0 = mai bine)
* `c_uci` / `c_npmi`: asociere PMI / PMI normalizat pe fereastră (mai mare = mai bine)
* `diversity`: cuvinte unice / cuvinte totale din toate topicurile (1.0 = niciun cuvânt repetat între topicuri; măsoară redundanța)

Cu doar 30 de documente, diferențele mici sunt zgomot.

In [9]:
def coherence_report(topics):
    topics = [[w for w in t if w in dictionary.token2id] for t in topics]
    scores = {}
    for m in ("c_v", "u_mass", "c_uci", "c_npmi"):
        cm = CoherenceModel(topics=topics, texts=chunk_texts, dictionary=dictionary,
                            coherence=m, processes=1)
        scores[m] = float(np.mean(cm.get_coherence_per_topic()))
    return scores


def topic_diversity(topics):
    words = [w for t in topics for w in t]
    return len(set(words)) / len(words)


rows = []
for name, res in results.items():
    tw = topic_words(res)
    rows.append({"model": name, **coherence_report(tw), "diversity": topic_diversity(tw)})
coh = pd.DataFrame(rows).set_index("model")
coh.round(3)

,c_v,u_mass,c_uci,c_npmi,diversity
model,,,,,
"NMF (BoW, k=3)",0.702,-3.003,-0.856,0.059,1.0
"NMF (TF-IDF, k=3)",0.696,-5.907,-4.251,-0.092,1.0
"NMF (BoW, k=5)",0.723,-2.576,-0.766,0.069,0.9
"NMF (TF-IDF, k=5)",0.766,-4.243,-2.508,0.013,0.9
"LDA (BoW, k=3)",0.654,-3.543,-1.428,0.025,1.0
"LDA (TF-IDF, k=3)",0.484,-10.271,-7.657,-0.228,1.0
"LDA (BoW, k=5)",0.657,-3.682,-1.563,0.040,0.9
"LDA (TF-IDF, k=5)",0.383,-15.566,-11.320,-0.372,0.4


In [10]:
# Tabel final (coerență + atribuire) și export pentru unirea cu rezultatele lui A
final = coh.join(assign_summary)
final.to_csv(OUT_DIR / "results_B.csv")
(OUT_DIR / "nmf_lda_topics_B.json").write_text(
    json.dumps({n: topic_words(r) for n, r in results.items()}, indent=2), encoding="utf-8")

# comparație directă cu LSA (persoana A)
res_a = pd.read_csv(IN_DIR / "results_A.csv", index_col=0)
cols = ["c_v", "u_mass", "diversity", "purity (abs)", "ARI (abs)"]
all_models = pd.concat([res_a[cols], final.rename(columns={"purity": "purity (abs)", "ARI": "ARI (abs)"})[cols]])
all_models.round(3)

,c_v,u_mass,diversity,purity (abs),ARI (abs)
model,,,,,
"LSA (BoW, k=3)",0.600,-4.848,0.90,0.933,0.808
"LSA (TF-IDF, k=3)",0.529,-7.873,0.90,1.000,1.000
"LSA (BoW, k=5)",0.541,-6.213,0.68,0.967,0.745
"LSA (TF-IDF, k=5)",0.640,-6.266,0.86,1.000,0.785
"NMF (BoW, k=3)",0.702,-3.003,1.00,1.000,1.000
"NMF (TF-IDF, k=3)",0.696,-5.907,1.00,1.000,1.000
"NMF (BoW, k=5)",0.723,-2.576,0.90,1.000,0.870
"NMF (TF-IDF, k=5)",0.766,-4.243,0.90,1.000,0.708
"LDA (BoW, k=3)",0.654,-3.543,1.00,1.000,1.000


# Raport de Analiză și Evaluare: NMF și LDA
*Partea 4 a proiectului: doar metodele persoanei B (NMF, LDA). Comparația cu LSA și concluziile comune se completează la întâlnirea cu persoana A.*

Tabelul de referință este cel din ultima celulă (`all_models`). Corpus foarte mic (30 de documente, 3 domenii), deci diferențele mici de scor sunt zgomot.

---

## 1. Cum influențează alegerea BoW vs TF-IDF topicurile extrase de NMF și LDA?

* **NMF este robust la reprezentare.** La k=3, ambele matrici dau aceleași 3 topicuri curate (romani / animale / energie verde), puritate 1.0 și ARI 1.0. TF-IDF schimbă doar cuvintele din topic: scoate cuvinte frecvente dar generale (`specie`, `male`, `prey` în BoW) și le înlocuiește cu cuvinte specifice (`tiger`, `dolphin`, `panda` pentru animale; `army`, `military` pentru romani). La k=5, TF-IDF produce subtopicuri mai fine (de ex. Imperiul Roman separat în *istorie generală* și *împărații Nerva–Antonini*: `hadrian trajan marcus antoninus nerva`).
* **LDA depinde puternic de reprezentare.** Cu BoW (numărări întregi, pentru care este definit modelul) LDA găsește aceleași 3 domenii (puritate 1.0, ARI 1.0). Cu TF-IDF rezultatul se degradează: la k=3 puritatea scade la 0.77 (ARI 0.50), iar topicul 2 amestecă `energy solar wind` cu `lion wolf`. La k=5 modelul **colapsează**: 4 din 5 topicuri au exact aceleași cuvinte de top (`antarctic pygmy orca chilean...`, adică cuvinte rare, alese practic la întâmplare), toate cele 30 de documente ajung în același topic (puritate 0.33, ARI 0.0, diversitate 0.4).
* **De ce?** LDA presupune numărări (distribuții multinomiale). TF-IDF produce valori fracționare, mici, și aplatizează diferențele dintre cuvinte, deci priorul și actualizările variaționale nu mai au semnal; topicurile nefolosite rămân aproape uniforme și afișează orice cuvânt. NMF minimizează doar o eroare de reconstrucție (Frobenius) și nu are ipoteze despre scara datelor, iar TF-IDF îi este chiar favorabil (cuvintele specifice domeniului au greutate mare).

---

## 2. Care metodă produce cele mai interpretabile topicuri? (pentru NMF/LDA; decizia finală împreună cu A)

* Din metodele mele, **NMF** este cea mai interpretabilă și cea mai stabilă: toate cele 4 configurații au puritate 1.0 (cu atribuire `argmax`), topicuri coerente și diversitate 0.9–1.0, indiferent de matrice.
* **LDA (BoW)** este aproape la fel de bun la k=3 și are avantajul interpretării probabilistice (fiecare document are o distribuție peste topicuri, fiecare topic o distribuție peste cuvinte), dar este mai sensibil: apar cuvinte intruse (`gorilla` în topicul de biocombustibili la k=5) și nu poate fi folosit în siguranță pe TF-IDF.
* Cea mai bună combinație pentru interpretabilitate din partea B: **NMF + TF-IDF, k=3** (cuvinte specifice, 3 grupuri de câte 10 documente, 0 erori).

---

## 3. Un scor Cv mai mare înseamnă întotdeauna topicuri mai utile?

**Nu.** Exemple din rezultatele mele:

* **`NMF (TF-IDF, k=5)`** are cel mai mare Cv din tabel (**0.766**), dar ARI față de cele 3 domenii este doar **0.71** (față de 1.0 la k=3 cu Cv=0.696): domeniile sunt împărțite în subtopicuri (energie în *solar/hidrogen* vs *eolian/mareic*, romani în două). Topicurile sunt coerente local, dar nu corespund domeniilor cerute; utilitatea depinde de scop.
* Cv favorizează topicuri cu cuvinte foarte apropiate în text: topicuri înguste și redundante pot scoate scoruri mari.
* Metricile Gensim nu sunt de acord între ele: la NMF k=3, BoW are un u_mass (-3.0) și un c_npmi (0.059) mult mai bune decât TF-IDF (-5.9, -0.092), deși Cv aproape egal (0.702 vs 0.696) și ambele modele au partiție perfectă.
* Invers, Cv detectează corect eșecul LDA TF-IDF (0.38–0.48, u_mass -10 până la -15.6), deci este util ca semnal de alarmă, dar nu ca criteriu unic. Concluzie: Cv trebuie citit împreună cu lista de cuvinte, puritatea/ARI și diversitatea.

---

## 4. Cum afectează creșterea numărului de topicuri coerența și redundanța?

| Model | Cv k=3 → k=5 | diversitate k=3 → k=5 | ARI k=3 → k=5 |
|---|---|---|---|
| NMF (BoW) | 0.702 → 0.723 | 1.0 → 0.9 | 1.0 → 0.870 |
| NMF (TF-IDF) | 0.696 → 0.766 | 1.0 → 0.9 | 1.0 → 0.708 |
| LDA (BoW) | 0.654 → 0.657 | 1.0 → 0.9 | 1.0 → 0.766 |
| LDA (TF-IDF) | 0.484 → 0.383 | 1.0 → 0.4 | 0.498 → 0.0 |

* **NMF:** Cv crește la k=5 (topicuri mai înguste, cuvinte mai apropiate), dar apar cuvinte repetate între topicuri (diversitate 0.9, ex. `bear`, `polar`, `male`, `prey` apar în mai multe topicuri de animale) și partiția nu mai corespunde celor 3 domenii. Topicurile în plus sunt **subteme reale**, nu zgomot: animale → *lup/haită*, *urs polar/gheață*, *felide*; energie → *solar/hidrogen* vs *eolian/mareic*.
* **LDA (BoW):** Cv aproape neschimbat, diversitate 0.9; apar topicuri mai puțin curate (animalele se împart în *mamifere marine/urs* și *prădători terestri*, iar în topicul de hidrogen intră `gorilla`).
* **LDA (TF-IDF):** k=5 este un caz extrem de **redundanță/colaps**: 4 topicuri identice și goale.
* Concluzie: peste numărul real de domenii (3), topicurile în plus fie împart un domeniu în subteme (NMF), fie devin redundante sau instabile (LDA). Un k mai mare nu înseamnă automat topicuri mai bune.

---

## 5. Corespund topicurile extrase celor trei domenii originale?

* **k=3: da, pentru NMF (ambele matrici) și LDA (BoW).** Fiecare topic conține exact cele 10 articole ale unui domeniu (matrice de confuzie diagonală, puritate 1.0, ARI 1.0, grupuri de 10), iar cuvintele de top se citesc direct ca *Imperiul Roman*, *animale*, *energie verde*. Excepție: **LDA (TF-IDF, k=3)** (puritate 0.77): doar articolele despre Roma (10/10) și energie (10/10) sunt grupate corect, în timp ce cele 10 articole *animals* se împart pe toate cele 3 topicuri (2/3/5), iar 5 dintre ele ajung în topicul de energie, ale cărui cuvinte de top amestecă `energy solar wind` cu `lion wolf`.
* **k=5:** puritatea rămâne 1.0 la NMF și LDA (BoW) (niciun topic nu amestecă domenii), dar domeniile se fragmentează, deci ARI scade (0.71–0.87 la NMF, 0.77 la LDA BoW). Diferențele față de cele 3 domenii sunt **subdomenii**, nu erori: animalele (cel mai eterogen domeniu: mamifere, păsări, delfini, balene, urși) se împart cel mai mult; energia în *solară* vs *eoliană/mareică*.
* **Observație despre corpus:** domeniile au vocabular foarte distinct (Roma antică / zoologie / energie), de aceea separarea este ușoară. Pe domenii apropiate (ex. energie solară vs eoliană) topicurile ar fi mult mai greu de recuperat.
